# Phase 15 — Balanced Pair Generation v2 and Leakage-Safe Splits

This notebook materializes a balanced `pairs_v2` job-fit dataset from frozen source snapshots and Phase 14 feature rules. It keeps pair generation notebook-first, writes production-training evidence under `reports/`, and rejects obvious split leakage before later baseline or model training notebooks can use the data.

Generated artifacts:

- `artifacts/pairs_v2.parquet`
- `reports/phase_15_balanced_pair_generation_splits.json`
- `reports/phase_15_pair_distribution_diagnostics.json`
- `reports/phase_15_leakage_report.json`

## Purpose
Document and verify Phase 15 — Balanced Pair Generation v2 and Leakage-Safe Splits in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 15.balanced.pair.generation.splits notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 15 — Balanced Pair Generation v2 and Leakage-Safe Splits.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Step 15.1 — Pair generation cells

### Purpose
Generate balanced pair types for job-fit training: high-fit positives, medium-fit pairs, hard negatives, random negatives, same-role different-seniority pairs, and cross-role confusing pairs.

### Required input
Frozen job/profile CSV snapshots from `legacy/dataset/`, Phase 14 normalization expectations, deterministic seed, and pair-generation config.

### Action
Load source rows, normalize skills, roles, language, and experience, then select deterministic profile/job combinations for every required pair type in train, validation, and test splits.

### Expected output
Candidate pairs covering all required pair types with label components and deterministic profile-level split assignment.

### Verification
Every required pair type is present overall and in validation/test, or a blocker is recorded in the report.


In [6]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PHASE_ID = "phase_15_balanced_pair_generation_splits"
PAIR_SCHEMA_VERSION = "pairs-v2-balanced-splits-v1"
SEED = 202615
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

REQUIRED_PAIR_TYPES = [
    "high_fit_positive",
    "medium_fit",
    "hard_negative",
    "random_negative",
    "same_role_different_seniority",
    "cross_role_confusing",
]
SPLIT_TARGETS = {"train": 420, "validation": 90, "test": 90}
HIGH_FIT_MIN_BY_EVAL_SPLIT = {"validation": 50, "test": 50}
SCORE_BANDS = {
    "low": (0.0, 0.399999),
    "medium": (0.4, 0.699999),
    "high": (0.7, 1.0),
}

SKILL_ALIASES = {
    "js": "javascript", "javascript": "javascript", "typescript": "typescript", "ts": "typescript",
    "reactjs": "react", "react.js": "react", "react js": "react", "react": "react",
    "nextjs": "next.js", "next.js": "next.js", "nodejs": "node.js", "node.js": "node.js", "node js": "node.js",
    "vuejs": "vue.js", "vue.js": "vue.js", "python": "python", "py": "python", "java": "java",
    "golang": "go", "go": "go", "laravel": "laravel", "php": "php", "sql": "sql", "mysql": "mysql",
    "postgres": "postgresql", "postgresql": "postgresql", "mongo": "mongodb", "mongodb": "mongodb",
    "aws": "aws", "gcp": "gcp", "google cloud": "gcp", "docker": "docker", "kubernetes": "kubernetes", "k8s": "kubernetes",
    "machine learning": "machine learning", "ml": "machine learning", "ai": "artificial intelligence", "artificial intelligence": "artificial intelligence",
    "analisis data": "data analysis", "data analysis": "data analysis", "data analytics": "data analysis",
    "project management": "project management", "manajemen proyek": "project management",
    "html": "html", "css": "css", "tailwind": "tailwind", "jquery": "jquery", "c++": "c++", "c#": "c#",
    "linux": "linux", "ci/cd": "ci/cd", "ci cd": "ci/cd", "tensorflow": "tensorflow", "pytorch": "pytorch",
}
ROLE_ALIASES = {
    "frontend": "frontend", "front end": "frontend", "mobile": "mobile", "android": "mobile", "ios": "mobile",
    "backend": "backend", "back end": "backend", "api": "backend", "server": "backend", "fullstack": "fullstack", "full stack": "fullstack",
    "data": "data", "machine learning": "data", "ai": "data", "cloud": "cloud", "devops": "cloud", "security": "security",
    "qa": "quality_assurance", "quality": "quality_assurance", "web": "web", "developer": "software_engineering", "engineer": "software_engineering",
}
EXPERIENCE_VALUES = {
    "Fresher": {"years": 0.0, "band": "entry"},
    "1-2 years": {"years": 1.5, "band": "junior"},
    "3-5 years": {"years": 4.0, "band": "mid"},
    "5+ years": {"years": 5.0, "band": "senior"},
    "ENTRY_LEVEL": {"years": 0.0, "band": "entry"},
    "JUNIOR": {"years": 1.5, "band": "junior"},
    "MID_LEVEL": {"years": 4.0, "band": "mid"},
    "SENIOR": {"years": 5.0, "band": "senior"},
    "LEAD": {"years": 6.0, "band": "lead"},
    "MANAGER": {"years": 7.0, "band": "manager"},
}
ALLOWED_LANGUAGES = {"ID", "EN", "MIXED", "UNKNOWN"}
ID_MARKERS = {"dan", "yang", "dengan", "untuk", "pengalaman", "keahlian", "minimal", "tahun", "kerja", "kemampuan"}
EN_MARKERS = {"and", "with", "for", "experience", "skills", "minimum", "years", "work", "ability", "requirements"}


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "training" / "notebooks").exists():
            return candidate
    raise RuntimeError("Repository root not found. Start notebook inside bisakerja-model repository.")

REPO_ROOT = find_repo_root()
REPORTS_DIR = REPO_ROOT / "reports"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
REPORTS_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)
JOBS_PATH = REPO_ROOT / "legacy/dataset/indotech_job_cleaned.csv"
PROFILES_PATH = REPO_ROOT / "legacy/dataset/techtalent_profile_cleaned.csv"
PAIRS_V2_PATH = ARTIFACTS_DIR / "pairs_v2.parquet"


def rel(path: Path) -> str:
    return str(path.resolve().relative_to(REPO_ROOT))


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_json(value: Any) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode("utf-8")).hexdigest()


def stable_int(*parts: Any) -> int:
    digest = hashlib.sha256("::".join(map(str, parts)).encode("utf-8")).hexdigest()
    return int(digest[:16], 16)


def stable_split(profile_id: Any) -> str:
    bucket = stable_int("split", profile_id) % 100
    if bucket < 70:
        return "train"
    if bucket < 85:
        return "validation"
    return "test"


def clean_token(token: Any) -> str:
    token = "" if token is None else str(token).lower().strip()
    token = token.replace("react.js", "react js").replace("node.js", "node js").replace("next.js", "nextjs")
    token = re.sub(r"[^a-z0-9+#./ -]+", " ", token)
    token = re.sub(r"\s+", " ", token).strip(" -_/.,")
    return token


def parse_skills(value: Any) -> tuple[str, ...]:
    text = "" if value is None or (isinstance(value, float) and math.isnan(value)) else str(value)
    raw_tokens = re.split(r"[,|;/]+|\s\|\s", text)
    normalized = []
    for raw in raw_tokens:
        token = clean_token(raw)
        if not token:
            continue
        canonical = SKILL_ALIASES.get(token, token)
        normalized.append(canonical)
    return tuple(sorted(set(normalized)))


def role_family(*values: Any) -> str:
    text = " ".join("" if value is None else str(value).lower() for value in values)
    families = [family for marker, family in ROLE_ALIASES.items() if marker in text]
    if not families:
        return "other"
    priority = ["backend", "frontend", "mobile", "fullstack", "data", "cloud", "security", "quality_assurance", "web", "software_engineering"]
    return sorted(families, key=lambda f: priority.index(f) if f in priority else 99)[0]


def normalize_experience(value: Any) -> dict[str, Any]:
    raw = "" if value is None else str(value).strip()
    if raw in EXPERIENCE_VALUES:
        return {"raw": raw, **EXPERIENCE_VALUES[raw], "known": True}
    return {"raw": raw, "years": 0.0, "band": "unknown", "known": not bool(raw)}


def normalize_language(value: Any, text: str = "") -> str:
    raw = "" if value is None else str(value).strip().upper()
    if raw in ALLOWED_LANGUAGES:
        return raw
    tokens = set(re.findall(r"[a-zA-Z]+", text.lower()))
    id_hits = len(tokens & ID_MARKERS)
    en_hits = len(tokens & EN_MARKERS)
    if id_hits and en_hits:
        return "MIXED"
    if id_hits:
        return "ID"
    if en_hits:
        return "EN"
    return "UNKNOWN"


def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 0.0
    union = a | b
    return len(a & b) / len(union) if union else 0.0


def requirement_coverage(profile_skills: set[str], job_skills: set[str]) -> float:
    return len(profile_skills & job_skills) / len(job_skills) if job_skills else 0.0


def experience_match(profile_years: float, job_years: float) -> float:
    gap = abs(profile_years - job_years)
    return max(0.0, 1.0 - min(gap, 6.0) / 6.0)


def score_band(score: float) -> str:
    if score >= 0.70:
        return "high"
    if score >= 0.40:
        return "medium"
    return "low"


def fit_score(pair_type: str, skill_overlap: float, req_cov: float, role_match: float, exp_match: float) -> float:
    base = (0.42 * skill_overlap) + (0.28 * req_cov) + (0.20 * role_match) + (0.10 * exp_match)
    adjustment = {
        "high_fit_positive": 0.22,
        "medium_fit": 0.00,
        "hard_negative": -0.18,
        "random_negative": -0.24,
        "same_role_different_seniority": -0.10,
        "cross_role_confusing": -0.08,
    }[pair_type]
    if pair_type == "high_fit_positive" and skill_overlap >= 0.25 and req_cov >= 0.35:
        base = max(base, 0.72 + 0.18 * min(1.0, (skill_overlap + req_cov) / 2.0))
    if pair_type == "medium_fit":
        base = 0.40 + 0.25 * min(1.0, base)
    if pair_type in {"hard_negative", "random_negative"}:
        base = min(base, 0.35)
    return round(float(min(1.0, max(0.0, base + adjustment))), 4)

jobs_raw = pd.read_csv(JOBS_PATH)
profiles_raw = pd.read_csv(PROFILES_PATH)

jobs = []
for _, row in jobs_raw.iterrows():
    skills = set(parse_skills(row.get("skills_clean", ""))) | set(parse_skills(row.get("requirements_concat", "")))
    exp = normalize_experience(row.get("experience_level", ""))
    text = " ".join(str(row.get(col, "")) for col in ["title", "normalized_title", "category", "description", "requirements_concat"])
    jobs.append({
        "job_id": str(row.get("job_id")),
        "skills": skills,
        "role_family": role_family(row.get("title", ""), row.get("normalized_title", ""), row.get("category", "")),
        "experience_years": float(exp["years"]),
        "experience_band": exp["band"],
        "language": normalize_language(row.get("language_signal", ""), text),
        "status": str(row.get("status", "")),
        "title": str(row.get("title", "")),
    })

profiles = []
for _, row in profiles_raw.iterrows():
    skills = set(parse_skills(row.get("Skills", ""))) | set(parse_skills(row.get("Required_Skills", "")))
    exp = normalize_experience(row.get("Experience", ""))
    profiles.append({
        "profile_id": str(row.get("ID")),
        "skills": skills,
        "role_family": role_family(row.get("Job_Role", "")),
        "experience_years": float(exp["years"]),
        "experience_band": exp["band"],
        "language": "UNKNOWN",
        "split": stable_split(row.get("ID")),
        "job_role": str(row.get("Job_Role", "")),
    })

jobs_by_role: dict[str, list[dict[str, Any]]] = defaultdict(list)
for job in jobs:
    jobs_by_role[job["role_family"]].append(job)
profiles_by_split = {split: [p for p in profiles if p["split"] == split] for split in SPLIT_TARGETS}
source_summary = {
    "jobs": {"path": rel(JOBS_PATH), "row_count": len(jobs), "sha256": sha256_file(JOBS_PATH)},
    "profiles": {"path": rel(PROFILES_PATH), "row_count": len(profiles), "sha256": sha256_file(PROFILES_PATH)},
}
print(source_summary)


{'jobs': {'path': 'legacy/dataset/indotech_job_cleaned.csv', 'row_count': 2073, 'sha256': '9ab27d2f3ee2e3e1269b28ddd865eddb2dd629113b05c51d2c4d4c3288dcf565'}, 'profiles': {'path': 'legacy/dataset/techtalent_profile_cleaned.csv', 'row_count': 69929, 'sha256': '79ec1cda8d3c7fef86566e02171085910ed0a1c725acfbb1f8cc4c04a5512494'}}


## Step 15.2 — `pairs_v2.parquet` materialization

### Purpose
Write a versioned pair dataset with required metadata for job-fit model training and later baseline evaluation.

### Required input
Generated pair candidates, source hashes, schema version, and label component functions.

### Action
Build deterministic `pair_id` values, score each pair, assign score bands, and export `artifacts/pairs_v2.parquet`.

### Expected output
A Parquet dataset containing `pair_id`, `profile_id`, `job_id`, `pair_type`, `split`, `score_band`, `job_fit_score`, label components, language, role family, and experience band.

### Verification
Required columns exist, IDs are non-null, scores are bounded, and duplicate `pair_id` values are rejected.


In [7]:
def candidate_rank(profile: dict[str, Any], job: dict[str, Any], pair_type: str) -> tuple[float, dict[str, float]]:
    skill_overlap = jaccard(profile["skills"], job["skills"])
    req_cov = requirement_coverage(profile["skills"], job["skills"])
    role = 1.0 if profile["role_family"] == job["role_family"] else 0.0
    exp = experience_match(profile["experience_years"], job["experience_years"])
    exp_gap = abs(profile["experience_years"] - job["experience_years"])
    if pair_type == "high_fit_positive":
        rank = (role * 2.5) + (skill_overlap * 3.0) + (req_cov * 2.0) + exp
    elif pair_type == "medium_fit":
        rank = -abs(skill_overlap - 0.28) + (role * 0.5) - abs(exp - 0.75) * 0.25
    elif pair_type == "hard_negative":
        rank = (role * 1.5) - skill_overlap - req_cov + (1.0 - exp)
    elif pair_type == "random_negative":
        rank = (1.0 - role) + (1.0 - skill_overlap) + (1.0 - req_cov)
    elif pair_type == "same_role_different_seniority":
        rank = (role * 2.0) + min(exp_gap, 6.0) / 6.0 + skill_overlap * 0.2
    elif pair_type == "cross_role_confusing":
        rank = (1.0 - role) + min(skill_overlap, 0.5) + min(req_cov, 0.5)
    else:
        rank = 0.0
    return rank, {
        "skill_overlap": skill_overlap,
        "requirement_coverage": req_cov,
        "role_match": role,
        "experience_match": exp,
        "experience_gap_years": round(exp_gap, 4),
    }


def candidate_pool(profile: dict[str, Any], pair_type: str) -> list[dict[str, Any]]:
    same_role = jobs_by_role.get(profile["role_family"], [])
    if pair_type in {"high_fit_positive", "medium_fit", "hard_negative", "same_role_different_seniority"} and same_role:
        return same_role
    if pair_type == "cross_role_confusing":
        return [job for job in jobs if job["role_family"] != profile["role_family"]]
    return jobs


def choose_job(profile: dict[str, Any], pair_type: str, used_pairs: set[tuple[str, str]]) -> tuple[dict[str, Any] | None, dict[str, float] | None]:
    pool = candidate_pool(profile, pair_type)
    ranked = []
    for job in pool:
        key = (profile["profile_id"], job["job_id"])
        if key in used_pairs:
            continue
        rank, components = candidate_rank(profile, job, pair_type)
        ranked.append((rank, stable_int(profile["profile_id"], job["job_id"], pair_type), job, components))
    if not ranked:
        return None, None
    ranked.sort(key=lambda item: (item[0], -item[1]), reverse=True)
    return ranked[0][2], ranked[0][3]

pairs: list[dict[str, Any]] = []
used_pairs: set[tuple[str, str]] = set()
blockers: list[dict[str, Any]] = []

for split, per_type_target in SPLIT_TARGETS.items():
    split_profiles = sorted(profiles_by_split[split], key=lambda p: stable_int(split, p["profile_id"]))
    for pair_type in REQUIRED_PAIR_TYPES:
        produced = 0
        for profile in split_profiles:
            if produced >= per_type_target:
                break
            job, components = choose_job(profile, pair_type, used_pairs)
            if job is None or components is None:
                continue
            score = fit_score(pair_type, components["skill_overlap"], components["requirement_coverage"], components["role_match"], components["experience_match"])
            # Keep pair-type score intent explicit even when source data is weak.
            if pair_type == "high_fit_positive" and score < 0.70:
                continue
            if pair_type == "medium_fit" and not (0.40 <= score < 0.70):
                continue
            if pair_type in {"hard_negative", "random_negative"} and score >= 0.40:
                continue
            pair_id = hashlib.sha256(f"{PAIR_SCHEMA_VERSION}:{profile['profile_id']}:{job['job_id']}:{pair_type}".encode("utf-8")).hexdigest()[:24]
            matched_skills = sorted(profile["skills"] & job["skills"])
            missing_skills = sorted(job["skills"] - profile["skills"])
            pair = {
                "pair_id": pair_id,
                "profile_id": profile["profile_id"],
                "job_id": job["job_id"],
                "pair_type": pair_type,
                "split": split,
                "score_band": score_band(score),
                "job_fit_score": score,
                "skill_overlap": round(components["skill_overlap"], 6),
                "requirement_coverage": round(components["requirement_coverage"], 6),
                "role_match": round(components["role_match"], 6),
                "experience_match": round(components["experience_match"], 6),
                "experience_gap_years": components["experience_gap_years"],
                "language": job["language"],
                "role_family": job["role_family"],
                "profile_role_family": profile["role_family"],
                "job_experience_band": job["experience_band"],
                "profile_experience_band": profile["experience_band"],
                "experience_band": f"profile:{profile['experience_band']}|job:{job['experience_band']}",
                "matched_skill_count": len(matched_skills),
                "missing_skill_count": len(missing_skills),
                "matched_skills": json.dumps(matched_skills[:25]),
                "missing_skills": json.dumps(missing_skills[:25]),
                "label_version": "weak-label-balanced-v2",
                "schema_version": PAIR_SCHEMA_VERSION,
            }
            pairs.append(pair)
            used_pairs.add((profile["profile_id"], job["job_id"]))
            produced += 1
        if produced < per_type_target:
            blockers.append({"split": split, "pair_type": pair_type, "target": per_type_target, "produced": produced})

pairs_df = pd.DataFrame(pairs)
required_columns = [
    "pair_id", "profile_id", "job_id", "pair_type", "split", "score_band", "job_fit_score",
    "skill_overlap", "requirement_coverage", "role_match", "experience_match", "experience_gap_years",
    "language", "role_family", "experience_band",
]
missing_columns = [col for col in required_columns if col not in pairs_df.columns]
if missing_columns:
    raise AssertionError(f"Missing required pairs_v2 columns: {missing_columns}")
if pairs_df["pair_id"].isna().any() or pairs_df["profile_id"].isna().any() or pairs_df["job_id"].isna().any():
    raise AssertionError("Pair identifiers must not be null")
if pairs_df["pair_id"].duplicated().any():
    raise AssertionError("Duplicate pair_id values detected")
if not pairs_df["job_fit_score"].between(0.0, 1.0).all():
    raise AssertionError("job_fit_score must stay within 0-1")

pairs_df.to_parquet(PAIRS_V2_PATH, index=False)
pairs_artifact = {"path": rel(PAIRS_V2_PATH), "row_count": int(len(pairs_df)), "sha256": sha256_file(PAIRS_V2_PATH)}
print(pairs_artifact)
print(pairs_df.groupby(["split", "pair_type"]).size().unstack(fill_value=0))
print(pairs_df.groupby(["split", "score_band"]).size().unstack(fill_value=0))


{'path': 'artifacts/pairs_v2.parquet', 'row_count': 3600, 'sha256': '0876d3353a220dc5fe1f93654b4d7ae85a19fe9bd4fd9c132e11d9f47958cd8a'}
pair_type   cross_role_confusing  hard_negative  high_fit_positive  \
split                                                                
test                          90             90                 90   
train                        420            420                420   
validation                    90             90                 90   

pair_type   medium_fit  random_negative  same_role_different_seniority  
split                                                                   
test                90               90                             90  
train              420              420                            420  
validation          90               90                             90  
score_band  high   low  medium
split                         
test          90   359      91
train        420  1679     421
validation    90   360  

## Step 15.3 — Leakage-safe split checks

### Purpose
Ensure validation/test results are not inflated by profile leakage from training.

### Required input
`pairs_v2` with deterministic profile-level split assignment.

### Action
Group by `profile_id`, verify each profile appears in one split only, and write a leakage report.

### Expected output
`reports/phase_15_leakage_report.json` with split counts and leakage violations.

### Verification
The leakage violation count must be zero.


In [8]:
profile_split_counts = pairs_df.groupby("profile_id")["split"].nunique()
leaking_profiles = sorted(profile_split_counts[profile_split_counts > 1].index.astype(str).tolist())
leakage_report = {
    "schema_version": PAIR_SCHEMA_VERSION,
    "artifact": pairs_artifact,
    "profile_count": int(pairs_df["profile_id"].nunique()),
    "pair_count": int(len(pairs_df)),
    "split_pair_counts": {str(k): int(v) for k, v in pairs_df["split"].value_counts().sort_index().items()},
    "split_profile_counts": {str(k): int(v) for k, v in pairs_df.groupby("split")["profile_id"].nunique().sort_index().items()},
    "leaking_profile_count": len(leaking_profiles),
    "leaking_profiles_sample": leaking_profiles[:25],
    "passed": len(leaking_profiles) == 0,
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
(REPORTS_DIR / "phase_15_leakage_report.json").write_text(json.dumps(leakage_report, indent=2, sort_keys=True))
if leaking_profiles:
    raise AssertionError(f"Profile-level split leakage detected: {len(leaking_profiles)} profiles")
leakage_report


{'schema_version': 'pairs-v2-balanced-splits-v1',
 'artifact': {'path': 'artifacts/pairs_v2.parquet',
  'row_count': 3600,
  'sha256': '0876d3353a220dc5fe1f93654b4d7ae85a19fe9bd4fd9c132e11d9f47958cd8a'},
 'profile_count': 920,
 'pair_count': 3600,
 'split_pair_counts': {'test': 540, 'train': 2520, 'validation': 540},
 'split_profile_counts': {'test': 140, 'train': 634, 'validation': 146},
 'leaking_profile_count': 0,
 'leaking_profiles_sample': [],
 'passed': True,
 'generated_at': '2026-06-02T04:47:02.115696+00:00'}

## Step 15.4 — Evaluation split coverage gates

### Purpose
Ensure validation and test contain enough high-fit and score-band examples for later score-band evaluation.

### Required input
`pairs_v2` split, pair type, and score-band columns.

### Action
Count high-fit positives and low/medium/high score bands in each split, then fail only for hard data-contract issues. When source data cannot support a target pair type, record the blocker instead of silently passing.

### Expected output
Coverage gate details embedded in the Phase 15 summary report.

### Verification
Validation and test pass high-fit minimums, every split has low/medium/high bands, and required pair types exist in validation/test unless blocker details are present.


In [9]:
pair_type_counts = pairs_df.groupby(["split", "pair_type"]).size().unstack(fill_value=0)
score_band_counts = pairs_df.groupby(["split", "score_band"]).size().unstack(fill_value=0)
high_fit_counts = pairs_df[(pairs_df["pair_type"] == "high_fit_positive") & (pairs_df["score_band"] == "high")].groupby("split").size()
coverage_failures = []
for split in ["validation", "test"]:
    high_count = int(high_fit_counts.get(split, 0))
    if high_count < HIGH_FIT_MIN_BY_EVAL_SPLIT[split]:
        coverage_failures.append({"split": split, "check": "high_fit_minimum", "expected": HIGH_FIT_MIN_BY_EVAL_SPLIT[split], "actual": high_count})
    for band in ["low", "medium", "high"]:
        count = int(score_band_counts.get(band, pd.Series()).get(split, 0)) if band in score_band_counts.columns else 0
        if count == 0:
            coverage_failures.append({"split": split, "check": "score_band_present", "band": band, "actual": 0})
    for pair_type in REQUIRED_PAIR_TYPES:
        count = int(pair_type_counts.get(pair_type, pd.Series()).get(split, 0)) if pair_type in pair_type_counts.columns else 0
        if count == 0:
            coverage_failures.append({"split": split, "check": "pair_type_present", "pair_type": pair_type, "actual": 0})

coverage_report = {
    "high_fit_minimums": HIGH_FIT_MIN_BY_EVAL_SPLIT,
    "high_fit_counts": {str(k): int(v) for k, v in high_fit_counts.items()},
    "score_band_counts": score_band_counts.fillna(0).astype(int).to_dict(),
    "pair_type_counts": pair_type_counts.fillna(0).astype(int).to_dict(),
    "blockers": blockers,
    "failures": coverage_failures,
    "passed": not coverage_failures and not blockers,
}
if coverage_failures:
    raise AssertionError(f"Phase 15 coverage gates failed: {coverage_failures}")
coverage_report


{'high_fit_minimums': {'validation': 50, 'test': 50},
 'high_fit_counts': {'test': 90, 'train': 420, 'validation': 90},
 'score_band_counts': {'high': {'test': 90, 'train': 420, 'validation': 90},
  'low': {'test': 359, 'train': 1679, 'validation': 360},
  'medium': {'test': 91, 'train': 421, 'validation': 90}},
 'pair_type_counts': {'cross_role_confusing': {'test': 90,
   'train': 420,
   'validation': 90},
  'hard_negative': {'test': 90, 'train': 420, 'validation': 90},
  'high_fit_positive': {'test': 90, 'train': 420, 'validation': 90},
  'medium_fit': {'test': 90, 'train': 420, 'validation': 90},
  'random_negative': {'test': 90, 'train': 420, 'validation': 90},
  'same_role_different_seniority': {'test': 90,
   'train': 420,
   'validation': 90}},
 'blockers': [],
 'failures': [],
 'passed': True}

## Step 15.5 — Distribution diagnostics

### Purpose
Publish diagnostics that make score, pair type, role, language, experience, and split coverage visible before baselines and model training use `pairs_v2`.

### Required input
Final `pairs_v2` DataFrame and leakage/coverage reports.

### Action
Aggregate distributions, compare with legacy weak-label range, mark legacy `pairs.parquet` as non-production evidence, and write Phase 15 report files under `reports/`.

### Expected output
`reports/phase_15_pair_distribution_diagnostics.json` and `reports/phase_15_balanced_pair_generation_splits.json`.

### Verification
Diagnostics include every requested slice and the summary report marks all Phase 15 acceptance criteria as passed.


In [10]:
def count_table(*cols: str) -> dict[str, Any]:
    grouped = pairs_df.groupby(list(cols)).size().reset_index(name="count")
    records = []
    for row in grouped.to_dict("records"):
        records.append({str(k): (int(v) if isinstance(v, (np.integer,)) else v) for k, v in row.items()})
    return {"columns": list(cols), "records": records}

score_summary = pairs_df["job_fit_score"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_dict()
diagnostics = {
    "schema_version": PAIR_SCHEMA_VERSION,
    "artifact": pairs_artifact,
    "source_files": source_summary,
    "score_summary": {str(k): float(v) for k, v in score_summary.items()},
    "score_band_by_split": count_table("split", "score_band"),
    "pair_type_by_split": count_table("split", "pair_type"),
    "role_family_by_split": count_table("split", "role_family"),
    "language_by_split": count_table("split", "language"),
    "experience_band_by_split": count_table("split", "experience_band"),
    "pair_type_by_score_band": count_table("pair_type", "score_band"),
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
(REPORTS_DIR / "phase_15_pair_distribution_diagnostics.json").write_text(json.dumps(diagnostics, indent=2, sort_keys=True))

acceptance = {
    "validation_and_test_high_fit_minimums_met": not any(f.get("check") == "high_fit_minimum" for f in coverage_failures),
    "no_profile_id_in_multiple_splits": leakage_report["passed"],
    "required_pair_types_present_overall_and_eval": not any(f.get("check") == "pair_type_present" for f in coverage_failures) and not blockers,
    "legacy_pairs_v1_not_production_training_evidence": True,
}
summary_report = {
    "schema_version": PAIR_SCHEMA_VERSION,
    "phase_id": PHASE_ID,
    "status": "complete" if all(acceptance.values()) and coverage_report["passed"] else "blocked",
    "artifact": pairs_artifact,
    "source_files": source_summary,
    "pair_generation_config": {
        "seed": SEED,
        "required_pair_types": REQUIRED_PAIR_TYPES,
        "split_targets_per_pair_type": SPLIT_TARGETS,
        "score_bands": SCORE_BANDS,
        "high_fit_minimum_by_eval_split": HIGH_FIT_MIN_BY_EVAL_SPLIT,
        "label_version": "weak-label-balanced-v2",
    },
    "coverage": coverage_report,
    "leakage": leakage_report,
    "diagnostics_report": rel(REPORTS_DIR / "phase_15_pair_distribution_diagnostics.json"),
    "legacy_pairs_policy": {
        "legacy_artifact": "legacy/artifacts/pairs.parquet",
        "production_training_evidence": False,
        "reason": "Phase 15 materializes full-range pairs_v2 with split-safe metadata; legacy weak-label pairs remain historical baseline evidence only.",
    },
    "acceptance_criteria": acceptance,
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
(REPORTS_DIR / "phase_15_balanced_pair_generation_splits.json").write_text(json.dumps(summary_report, indent=2, sort_keys=True))
if summary_report["status"] != "complete":
    raise AssertionError(f"Phase 15 incomplete: {summary_report}")
summary_report


{'schema_version': 'pairs-v2-balanced-splits-v1',
 'phase_id': 'phase_15_balanced_pair_generation_splits',
 'status': 'complete',
 'artifact': {'path': 'artifacts/pairs_v2.parquet',
  'row_count': 3600,
  'sha256': '0876d3353a220dc5fe1f93654b4d7ae85a19fe9bd4fd9c132e11d9f47958cd8a'},
 'source_files': {'jobs': {'path': 'legacy/dataset/indotech_job_cleaned.csv',
   'row_count': 2073,
   'sha256': '9ab27d2f3ee2e3e1269b28ddd865eddb2dd629113b05c51d2c4d4c3288dcf565'},
  'profiles': {'path': 'legacy/dataset/techtalent_profile_cleaned.csv',
   'row_count': 69929,
   'sha256': '79ec1cda8d3c7fef86566e02171085910ed0a1c725acfbb1f8cc4c04a5512494'}},
 'pair_generation_config': {'seed': 202615,
  'required_pair_types': ['high_fit_positive',
   'medium_fit',
   'hard_negative',
   'random_negative',
   'same_role_different_seniority',
   'cross_role_confusing'],
  'split_targets_per_pair_type': {'train': 420, 'validation': 90, 'test': 90},
  'score_bands': {'low': (0.0, 0.399999),
   'medium': (0.4, 0.